# Projection V2 — Kaggle GPU training

Upload `projection-v2-training-inputs.zip` từ Bước 1 của notebook full-pipeline. Bật GPU T4 và Internet. Bundle chỉ chứa train/validation cho dataset trong config; không có test vectors hoặc response/credential. Mạng residual nhỏ đọc embedding cache, không tải CodeT5 lại. Xuất `projection-v2-heads.zip` về `bundles/` local.

In [ ]:
from pathlib import Path
import os,sys,subprocess
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
ROOT=Path('/kaggle/working/grace-projection-v2')
matches=sorted(Path('/kaggle/input').rglob('projection-v2-training-inputs.zip'))
if len(matches)!=1: raise RuntimeError('Upload đúng một projection-v2-training-inputs.zip hoặc gán INPUT_ZIP bằng đường dẫn cụ thể')
INPUT_ZIP=matches[0]
RESUME_ZIP=None  # Có thể gán đường dẫn projection-v2-training-checkpoint.zip của lượt trước.
EXTRA_SEEDS=[]  # [43,44] để kiểm tra replication offline; không chọn seed tốt nhất bằng test.


In [ ]:
import hashlib, json, sys, zipfile
from pathlib import Path, PurePosixPath
def bootstrap_sources(root, archive_path, expected_kind):
    root.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        manifest=json.loads(archive.read('bundle_manifest.json'))
        if manifest['schema']!='grace_projection_v2_bundle_v1' or manifest['kind']!=expected_kind:
            raise ValueError('Wrong V2 input bundle')
        for entry in manifest['entries']:
            name=entry['path'];p=PurePosixPath(name)
            source_file=name.startswith('grace_h1/') and p.suffix=='.py'
            dependency_file=name in ('requirements.txt','requirements-cpu.txt','requirements-training.txt')
            if not (source_file or dependency_file): continue
            if p.is_absolute() or '..' in p.parts or ':' in name or '\\' in name:
                raise ValueError('Unsafe source path')
            data=archive.read(name)
            if len(data)!=entry['bytes'] or hashlib.sha256(data).hexdigest()!=entry['sha256']:
                raise ValueError('Source checksum mismatch')
            target=root/name;target.parent.mkdir(parents=True,exist_ok=True)
            if target.exists() and target.read_bytes()!=data:
                raise ValueError('Source changed: use a fresh cloud ROOT')
            target.write_bytes(data)
    if str(root) not in sys.path: sys.path.insert(0,str(root))

bootstrap_sources(ROOT,INPUT_ZIP,'training_inputs')
subprocess.check_call([sys.executable,'-m','pip','install','-r',str(ROOT/'requirements-training.txt')])
from grace_h1.portable_v2 import import_bundle,create_bundle
print(import_bundle(ROOT,INPUT_ZIP,'training_inputs'))
if RESUME_ZIP:
    with zipfile.ZipFile(RESUME_ZIP) as z: resume_kind=json.loads(z.read('bundle_manifest.json'))['kind']
    if resume_kind not in ('heads','training_checkpoint'): raise ValueError('Resume archive must be a V2 training output')
    print(import_bundle(ROOT,RESUME_ZIP,resume_kind))
from grace_h1.experiment_v2 import load_config,train_all
from grace_h1.io import write_json
config=load_config(ROOT)
print(json.dumps(config,indent=2))


## Train và chọn checkpoint

Log mỗi epoch có macro label P@5 trên validation và overlap với teacher. Checkpoint `last.pt` được ghi mỗi epoch; interrupt chạy phần finally để đóng gói phần đã xong. `baseline_fallback` nghĩa là chưa tìm được learned checkpoint đạt các gate. CE control cùng kiến trúc giúp tách hiệu quả của loss local contrastive. Các số này chưa là benchmark GRACE + LLM.

In [ ]:
import copy
try:
    selected=train_all(ROOT,config)
    replications={str(config['training']['seed']):selected}
    for seed in EXTRA_SEEDS:
        replica=copy.deepcopy(config);replica['training']['seed']=int(seed)
        replications[str(seed)]=train_all(ROOT,replica)
    if EXTRA_SEEDS:
        write_json(ROOT/'artifacts/projection_v2/replications.json',replications)
        selected=train_all(ROOT,config)  # Khôi phục selection seed chính.
finally:
    if (ROOT/'artifacts/projection_v2').exists():
        print(create_bundle(ROOT,Path('/kaggle/working/projection-v2-training-checkpoint.zip'),'training_checkpoint'))
for d,heads in selected['heads'].items():
    for name,h in heads.items(): print(d,name,h['selection_status'],h['best'])


In [ ]:
print(create_bundle(ROOT,Path('/kaggle/working/projection-v2-heads.zip'),'heads'))
print('Tải projection-v2-heads.zip vào bundles/ local và quay lại full-pipeline.ipynb, Bước 1.')
